In [1]:
import tensorflow as tf
import TensorSlider as ts
import keras
import numpy as np
import DataPrep


2025-02-19 17:04:50.836498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1739981090.848567  221921 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1739981090.852297  221921 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-19 17:04:50.867605: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
tfrecordpath = "../Data/tfrecords/"

windowsize = 50
lookahead = 5
batch_size = 20

coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP", "BNBUSDT_PERP", "LINKUSD_PERP", "TRXUSD_PERP", "XLMUSD_PERP","DOTUSD_PERP"]
datasets = DataPrep.getAllSliders(coins, windowsize, lookahead, batch_size, tfrecordpath)
valdataset = datasets.pop(-1)


I0000 00:00:1739981093.152234  221921 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5592 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060 Ti, pci bus id: 0000:05:00.0, compute capability: 8.6


In [3]:
import keras
import os

def load_model(name, optimizer, fresh = False, export = False):
    """
    Load model and latest checkpoint (if applicable). Returns model, and last epoch that was trained.
    If no checkpoints present, either creates the checkpoint folder or trains directly on the saved model.
    """
    folder = "models/" + name + "/"
    # Load model
    model = keras.models.load_model(folder + "model.keras")

    checkpoint = tf.train.Checkpoint(optimizer=optimizer, model=model)
    manager = tf.train.CheckpointManager(checkpoint=checkpoint, directory=folder, max_to_keep=3)

    if not fresh:
        manager.restore_or_initialize()

    if export:
        model.export(folder)
        raise Exception("exported!")


    return model, manager

class saveEachEpoch(tf.keras.callbacks.Callback):
    """
    custom callback for saving checkpoints because of course we need to do this on our own
    """

    def __init__(self, checkpointmanager):
        super().__init__()
        # no idea if we want to or need to super this
        try:
            self.lastEpoch = int(checkpointmanager.latest_checkpoint.split("-")[-1])
        except Exception as e:
            print(e)
            self.lastEpoch = 0

        self.checkpointManager = checkpointmanager
        print("Model was trained for " + str(self.lastEpoch) + " epochs before.")

    def on_epoch_end(self, epoch, logs):
        """
        Create a checkpoint and save.
        """
        print(f"Epoch {self.lastEpoch} ended")
        # Increment which epoch this is
        self.lastEpoch += 1
        # Save weights
        self.checkpointManager.save(checkpoint_number=epoch)


In [4]:
#zip the different dataset sources
zipped = tf.data.Dataset.zip(datasets=tuple(datasets))
def combineZippedBatches(*zipped):
    batchshape = zipped[0][0]
    # Create first tensors to concat the rest
    data = zipped[0][0]
    label = zipped[0][1]
    for i in range(1, len(zipped)): # iterate over each remaining pair
        data = tf.concat([data, zipped[i][0]], axis=0)
        label = tf.concat([label, zipped[i][1]], axis=0)

    return data, label

@tf.function
def duplicateLabels(data, label):
    stretchamount = 3
    copied = []
    for i in range(stretchamount):
        copied.append(label)

    return data, tuple(copied)

# combine batches into one megabatch
batchTogether = zipped.map(combineZippedBatches, num_parallel_calls=tf.data.AUTOTUNE)
trainingset = batchTogether.map(DataPrep.createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE)

# duplicate labels
#trainingset = trainingset.map(duplicateLabels, num_parallel_calls=tf.data.AUTOTUNE)

valset = valdataset.map(DataPrep.createLabelsBatch)
#valset = valset.map(duplicateLabels, num_parallel_calls=tf.data.AUTOTUNE)

# Prefetch and create labels
training = trainingset.prefetch(tf.data.AUTOTUNE)
validation = valset.prefetch(tf.data.AUTOTUNE)

## Custom losses and metrics

In [5]:
def minmax_MSE(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true[:, 0:2] - y_pred[:,0:2]), axis=1)

def expected_MSE(y_true, y_pred):
    # Calculate mse of expected log variance
    return tf.reduce_mean(tf.square(y_pred[:,2:4]), axis=1)

def error_of_error_MSE(y_true, y_pred):
    return tf.abs(minmax_MSE(y_true, y_pred) - expected_MSE(y_true, y_pred))

def expected_error_variance(y_true, y_pred):
    exerrors = tf.reduce_mean(y_pred[:,2:4], axis=0)
    return tf.math.reduce_variance(exerrors)

def longableshortable(y_true, y_pred):
    minprofit = 0.02 # We want at least 0.2 percent profit, remember 1 = 10%

    # Check if the real minimum is below predicted min + error, and vice versa
    stoppedlong = tf.less_equal(y_true[:,0], y_pred[:,0] - y_pred[:,2])
    stoppedshort = tf.greater_equal(y_true[:,1], y_pred[:,1] + y_pred[:,3])

    # Check if real maximum is above predicted max - error, and vice versa
    longtp = tf.greater_equal(y_true[:,1], y_pred[:,1] - y_pred[:,3])
    shorttp = tf.less_equal(y_true[:,0], y_pred[:,0] + y_pred[:,2])

    # Compute predicted price delta
    # delta = max-error - min+error
    pricedelta = (y_pred[:,1] - y_pred[:,3]) - (y_pred[:,0] + y_pred[:,2])

    # Check if pricedelta is profitable
    profitable = tf.greater(pricedelta, minprofit)

    # long/short only true if stopped is false, it reaches take profit, and the trade would be profitable
    long = tf.logical_and(tf.logical_and(tf.logical_not(stoppedlong), profitable), longtp)
    short = tf.logical_and(tf.logical_and(tf.logical_not(stoppedshort), profitable), shorttp)

    return long, short

def correctDecision(y_true, y_pred):
    long, short = longableshortable(y_true, y_pred)

    # if both long and short, check which entry price closer. Entry price is min/max prediction
    conflicting = tf.math.logical_and(long, short)
    distancelong = tf.abs(y_true[:,0])
    distanceshort = tf.abs(y_true[:,1])

    # Where we have a conflict, pick the side closer to opening price
    resolvelong = tf.where(conflicting, tf.less_equal(distancelong, distanceshort), tf.greater(distancelong, distanceshort))
    resolveshort = tf.where(conflicting, tf.greater(distancelong, distanceshort), tf.less_equal(distancelong, distanceshort))

    # If the long is in conflicting, resolve long, otherwise long
    resultlong = tf.where(conflicting, resolvelong, long)
    resultshort = tf.where(conflicting, resolveshort, short)

    # neutral if neither long nor short
    resultneutral = tf.logical_not(tf.logical_or(resultlong, resultshort))


    result = tf.cast(tf.stack([resultlong, resultshort, resultneutral], axis=1), tf.int32)
    return result

def decision_loss(y_true, y_pred):

    result = correctDecision(y_true, y_pred)

    cross_entropy = keras.losses.CategoricalCrossentropy(reduction=None)
    cross_entropy_loss = cross_entropy(result, y_pred[:,4:7])
    return cross_entropy_loss

def tradeprofits(y_true, y_pred):

    result = correctDecision(y_true, y_pred)

    # now we have what should be true, lets check what is actually true
    # the truth is y_pred[;,4:7]
    # prediction is argmax of result
    trade = tf.argmax(y_pred[:,4:7], axis=1) # which index is predicted
    correct = tf.argmax(result, axis=1) # which index is correct
    profits = tf.where(trade == correct, 1, -1) # if index is correct, return 1, else return -1
    profits = tf.where(trade == 2, 0, profits) # if the index is 2, i.e. neutral, return 0
    return tf.cast(profits, tf.float32)

def amountNeutral(y_true, y_pred):
    return y_pred[:,6]

def isNeutral(y_true, y_pred):
    neutral = tf.argmax(y_pred[:,4:7], axis=1)
    return tf.where(neutral == 2, 1, 0)


def portionTradesProfitable(y_true, y_pred):
    """Percentage of predicted trades which are profitable"""
    # get the trade
    # 0 long, 1 short, 2 neutral
    trade = tf.argmax(y_pred[:,4:7], axis=1)
    longable, shortable = longableshortable(y_true, y_pred)

    # if longable, and predicted long, good long, and vice versa
    goodlong = tf.cast(tf.logical_and(longable, trade == 0), tf.float32)
    badlong = tf.cast(tf.logical_and(tf.logical_not(longable), trade == 0), tf.float32)
    goodshort = tf.cast(tf.logical_and(shortable, trade == 1), tf.float32)
    badshort = tf.cast(tf.logical_and(tf.logical_not(shortable), trade == 1), tf.float32)

    good = tf.reduce_sum(goodlong) + tf.reduce_sum(goodshort)
    bad = tf.reduce_sum(badlong) + tf.reduce_sum(badshort)

    return tf.keras.ops.nan_to_num(good/(good+bad))

def portionTradesUnProfitable(y_true, y_pred):
    """Percentage of predicted trades which are profitable"""
    # get the trade
    # 0 long, 1 short, 2 neutral
    trade = tf.argmax(y_pred[:,4:7], axis=1)
    longable, shortable = longableshortable(y_true, y_pred)

    # if longable, and predicted long, good long, and vice versa
    goodlong = tf.cast(tf.logical_and(longable, trade == 0), tf.float32)
    badlong = tf.cast(tf.logical_and(tf.logical_not(longable), trade == 0), tf.float32)
    goodshort = tf.cast(tf.logical_and(shortable, trade == 1), tf.float32)
    badshort = tf.cast(tf.logical_and(tf.logical_not(shortable), trade == 1), tf.float32)

    good = tf.reduce_sum(goodlong) + tf.reduce_sum(goodshort)
    bad = tf.reduce_sum(badlong) + tf.reduce_sum(badshort)

    return tf.keras.ops.nan_to_num(bad/(good+bad))

def portionNeutral(y_true, y_pred):
    """Percentage of predicted trades which are profitable"""
    # get the trade
    # 0 long, 1 short, 2 neutral
    trade = tf.argmax(y_pred[:,4:7], axis=1)
    neutral = tf.cast(tf.where(trade == 2, 1, 0), tf.float32)
    return tf.reduce_sum(neutral) / y_true.shape[0]

def combinedloss(y_true, y_pred):

    minmaxMSE = minmax_MSE(y_true, y_pred)
    errorMSE = 0.5 * error_of_error_MSE(y_true, y_pred)
    decisionloss = decision_loss(y_true, y_pred)
    #neutral = 1.2 * amountNeutral(y_true, y_pred) # incentivize less neutrality, but not too much
    #profits = 1.5 * tf.math.sigmoid(1 - tradeprofits(y_true, y_pred)) # higher proportion profitable trades -> less loss
    profitable = 1 - portionTradesProfitable(y_true, y_pred)
    profitable = tf.ones_like(minmaxMSE) * profitable

    #summed_loss = minmaxMSE + errorMSE + decisionloss + tf.square(profits) + tf.square(neutral)
    stacked_loss = tf.stack([minmaxMSE, errorMSE, decisionloss, profitable])
    return stacked_loss

def meancombinedloss(y_true, y_pred):
    minmaxMSE = minmax_MSE(y_true, y_pred)
    errorMSE = error_of_error_MSE(y_true, y_pred)
    decisionloss = decision_loss(y_true, y_pred)
    minmaxMSE = tf.reduce_mean(minmaxMSE)
    errorMSE = tf.reduce_mean(errorMSE)
    #profits = (1 - tf.reduce_mean(tradeprofits(y_true, y_pred)))  # higher proportion profitable trades -> less loss
    profitable = 1 - portionTradesProfitable(y_true, y_pred)
    unprofitable = portionTradesUnProfitable(y_true, y_pred)
    decisionloss = tf.reduce_mean(decisionloss)


    #stacked_loss = tf.stack([minmaxMSE, errorMSE, profits, tf.square(profitable), tf.square(decisionloss)])
    stacked_loss = tf.stack([minmaxMSE, 0.3 * errorMSE, decisionloss, 1.5 * profitable])#, tf.square(unprofitable)])
    return stacked_loss


In [ ]:
modelName = "dingus7"

tensorboard = keras.callbacks.TensorBoard(
                                                log_dir=f"models/{modelName}/logs",
                                                histogram_freq=100,
                                                write_graph=True,
                                                write_images=True,
                                                write_steps_per_second=True,
                                                update_freq="batch",
                                                #profile_batch = '70,100',
                                                embeddings_freq=0,
                                                embeddings_metadata=None,
                                            )
#optimizer = keras.optimizers.Adam(clipvalue=0.5)
optimizer = keras.optimizers.Adam(amsgrad = True, clipnorm=1, learning_rate=0.0005) # amsgrad seems to be required

model, manager = load_model(modelName, optimizer)

model.compile(loss=combinedloss,
                optimizer=optimizer,
                metrics=[expected_error_variance, minmax_MSE, error_of_error_MSE, decision_loss, amountNeutral, portionTradesProfitable, portionTradesUnProfitable,tradeprofits, portionNeutral, isNeutral],
              )

model.summary()

history = model.fit(training, epochs=20, verbose=0, validation_data=validation, callbacks=[tensorboard, saveEachEpoch(manager)])

/root/miniconda3/envs/levbot/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 74 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 6, 5, 50)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 1500)      │          0 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 2000)      │  3,002,000 │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 2000)      │          0 │ dense_12[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 2000)      │  4,002,000 │ dropout_10[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 2000)      │          0 │ dense_13[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 1500)      │  3,001,500 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 1500)      │          0 │ dense_14[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 1000)      │  1,501,000 │ dropout_12[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 1000)      │          0 │ dense_15[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 700)       │    700,700 │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 700)       │          0 │ dense_16[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 500)       │    350,500 │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 500)       │          0 │ dense_17[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 500)       │    250,500 │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 500)       │          0 │ dense_18[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 500)       │    250,500 │ dropout_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 500)       │          0 │ dense_19[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 250)       │    125,250 │ dropout_17[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ minmax (Dense)      │ (None, 2)         │        502 │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 13,185,707 (50.30 MB)

 Trainable params: 13,185,707 (50.30 MB)

...

Epoch 2 ended


2025-02-19 17:13:21.616801: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
2025-02-19 17:13:21.616852: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 16121537319903099656


Epoch 3 ended


2025-02-19 17:15:08.781340: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 16121537319903099656
2025-02-19 17:15:27.985336: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 16121537319903099656
2025-02-19 17:15:27.985382: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 2501331973236057824


Epoch 4 ended
